# MGS-7 : TSP combinatoire -- la grammaire de composition est agnostique à la représentation

[← MGS-6 Benchmarks](MGS-6-Benchmarks.ipynb) | [↑ Série MGS](README.md)

**Question centrale.** MGS-5 a montré que les métaheuristiques composées géométriques (WOA) se reconstruisent depuis des primitives sur des surfaces **continues** (gènes en `double`). Mais la grammaire de composition (`Match`, `Container`, `Scoped`, `Islands`) vit *sous* la couche géométrique. Cette couche fait-elle une hypothèse sur le type des gènes ?

Ce notebook le vérifie : la même grammaire structure la recherche sur un problème de **permutation** — le Voyageur de Commerce (TSP), où les gènes sont des index de villes entiers — **sans aucune adaptation**. Si les primitives de composition n'avaient pas été conçues de façon agnostique à la représentation (comme le sont les opérateurs de GeneticSharp : bit, permutation, arbre, flottant), on ne pourrait pas réutiliser `IslandMetaHeuristic` tel quel sur TSP. C'est « components over metaphors » en pratique : les briques ne présument pas du domaine.

In [1]:
// Wiring : charge MetaGeneticSharp + GeneticSharp + GeneticSharp.Extensions (TSP).
// Prérequis : dotnet build c:/dev/MetaGeneticSharp/MetaGeneticSharp.sln
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Infrastructure.Framework.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Domain.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Infrastructure.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Domain.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Extensions/bin/Debug/net9.0/MetaGeneticSharp.Extensions.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Extensions/bin/Debug/net9.0/GeneticSharp.Extensions.dll"

using MetaGeneticSharp;
using GeneticSharp;
using GeneticSharp.Extensions;

Console.WriteLine("DLLs chargées (incluant GeneticSharp.Extensions = TSP).");
Console.WriteLine($"  TspFitness          : {typeof(TspFitness).Name}");
Console.WriteLine($"  TspChromosome       : {typeof(TspChromosome).Name}");
Console.WriteLine($"  IslandMetaHeuristic : {typeof(IslandMetaHeuristic).Name}");
Console.WriteLine($"  MigrationMode       : {typeof(MigrationMode).Name}");

The below script needs to be able to find the current output cell; this is an easy method to get it.

DLLs chargées (incluant GeneticSharp.Extensions = TSP).


  TspFitness          : TspFitness


  TspChromosome       : TspChromosome


  IslandMetaHeuristic : IslandMetaHeuristic


  MigrationMode       : MigrationMode


## Le problème : 20 villes, gènes = index entiers

`TspFitness` génère N villes dans un carré et mesure la longueur totale du tour — les gènes sont l'ordre de visite, c'est-à-dire une permutation des index `0..N-1`. GeneticSharp *maximise* le fitness, donc `TspFitness.Evaluate` renvoie `1 - longueur/(N*1000)` (tour court ⇒ fitness élevé). La métrique qu'on surveille est la **longueur du tour** (`TspChromosome.Distance`), qu'on minimise.

On fixe le générateur aléatoire (`BasicRandomization.ResetSeed`) pour que les villes et les populations initiales soient reproductibles d'une configuration à l'autre — comparaison équitable.

In [2]:
// Reproductibilité : graine fixe. Les villes sont générées une fois (partagées entre configs).
void ResetRng(int seed) => BasicRandomization.ResetSeed(seed);

const int N = 20;
ResetRng(42);
var fitness = new TspFitness(N, 0, 100, 0, 100);
Console.WriteLine($"TSP : {N} villes chargées dans [0,100]².");

TSP : 20 villes chargées dans [0,100]².


## Un harness commun

Les trois configurations (baseline, îles, exercices) partagent le même moteur `MetaGeneticAlgorithm`. On factorise l'exécution dans un helper qui enregistre, à chaque génération, la **longueur du meilleur tour**. C'est ici que l'agnosticité se manifeste concrètement : le helper ne sait pas que les gènes sont des permutations — il pilote le moteur sur des `IChromosome` génériques et ne lit le type concret (`TspChromosome`) que pour extraire la métrique métier.

In [3]:
record RunResult(string Label, double FinalTourLength, double[] BestPerGen);

RunResult Run(string label, int generations, int popSize,
              ICrossover crossover, IMutation mutation, IMetaHeuristic meta)
{
    ResetRng(7); // population initiale reproductible : mêmes villes, même départ pour chaque config.
    var adam = new TspChromosome(N);
    var population = new MetaPopulation(popSize, popSize, adam);
    var ga = new MetaGeneticAlgorithm(population, fitness,
        new EliteSelection(), crossover, mutation, meta)
    {
        Termination = new GenerationNumberTermination(generations)
    };

    var best = new List<double>();
    ga.GenerationRan += (_, _) =>
    {
        // MetaPopulation est order-preserving : BestChromosome n'est pas peuplé au moment de l'event.
        // On déduit le meilleur depuis Chromosomes par fitness (toujours fixé par Evaluate).
        // Fitness max <=> tour le plus court ; ce chromosome a son .Distance fixé (même Evaluate).
        var bestChrom = ga.Population.CurrentGeneration.Chromosomes
            .OrderByDescending(c => c.Fitness).First();
        best.Add(((TspChromosome)bestChrom).Distance);
    };
    ga.Start();

    double final = best.Count > 0 ? best[^1] : double.NaN;
    return new RunResult(label, final, best.ToArray());
}

Console.WriteLine("Helper Run prêt.");

Helper Run prêt.


## Config 1 — baseline : GA classique

La `DefaultMetaHeuristic` reproduit le comportement GA standard : sélection élitiste → croisement (`OrderedCrossover`, OX1) → mutation (`ReverseSequenceMutation`) → réinsertion élitiste. Population panmictique : aucun partitionnement, tous les chromosomes interagissent.

In [4]:
var baseline = Run("Baseline (panmictic)", 100, 40,
    new OrderedCrossover(), new ReverseSequenceMutation(),
    new DefaultMetaHeuristic());
Console.WriteLine($"{baseline.Label,-26} tour final = {baseline.FinalTourLength:F1}");

Baseline (panmictic)       tour final = 440,8


## Config 2 — modèle insulaire sur permutations

Mêmes opérateurs, mais `IslandMetaHeuristic(10, 4, ...)` partitionne les 40 chromosomes en **4 îles de 10**. Chaque île évolue indépendamment ; la migration (mode `Static`, période 5) échange des chromosomes **entiers** entre îles. Point clé : **aucune arithmétique sur les gènes** — la migration déplace des permutations complètes, pas des positions réinterpolées. C'est précisément pourquoi le modèle fonctionne aussi bien sur des permutations que sur des vecteurs continus (MGS-4).

In [5]:
var islandMeta = new IslandMetaHeuristic(10, 4, new DefaultMetaHeuristic())
{
    MigrationMode = MigrationMode.Static,
    MigrationsGenerationPeriod = 5,
};
var island = Run("Islands (4 x 10, Static)", 100, 40,
    new OrderedCrossover(), new ReverseSequenceMutation(),
    islandMeta);
Console.WriteLine($"{island.Label,-26} tour final = {island.FinalTourLength:F1}");

Islands (4 x 10, Static)   tour final = 513,9


## Config 3 — un crossover géométrique sur les permutations (Moraglio)

La Lecture précédente (et MGS-5) présente les composés géométriques comme *typés* : WOA, EO font de l'arithmétique sur des `double`. C'est vrai pour leurs **opérateurs continus**. Mais la **théorie** du crossover géométrique (Moraglio & Poli, GECCO 2004 ; thèse Moraglio 2007) est plus générale : un crossover est *géométrique* si la progéniture se situe sur le **segment métrique** entre ses parents — et ce, dans **n'importe quel espace métrique**, pas seulement euclidien.

Pour les permutations, la métrique naturelle est la **distance de swap** (combien de transpositions séparent deux ordres). Le `GeometricCrossover<int>` du fork, muni de son `OrderedEmbedding<int>` (`IsOrdered = true`), réalise cela concrètement :

1. il mappe les parents (permutations d'indices de villes) vers l'espace métrique ;
2. applique l'opérateur géométrique (ici le centroïde par position) → une cible métrique ;
3. **remappe vers une permutation valide** en partant d'un clone du premier parent et en **marchant par swaps** vers la cible (`FlipGene`) — chaque transposition rapproche de la cible, la progéniture reste une permutation à chaque pas.

C'est l'incarnation du *geodesic crossover* de Moraglio : la progéniture est un point du segment swap-distance entre les parents. Le **même** `GeometricCrossover` qui servait aux composés continus (DE/WOA/EO/FBI) s'applique donc aux permutations **via l'embedding** — c'est l'embedding (la métrique), pas l'opérateur, qui porte la géométrie.

**Pivot deep learning géométrique.** Les indices de villes sont des *étiquettes* arbitraires, pas des coordonnées ordinales : le centroïde `(v₁ + v₂) / 2` n'a aucun sens géographique. Le deep learning géométrique (Bronstein *et al.*, *Geometric Deep Learning*) formalise cette intuition — un bon embedding doit être **équivariant par permutation** (aucun ordre de ville n'est privilégié) et refléter la **structure du problème** (ici, la géométrie euclidienne des villes dans `[0, 100]²`). Un embedding appris ou positionnel ferait mieux ; le centroïde naïf sur étiquettes est la **baseline** qui mesure précisément la limite que la théorie prédit. C'est l'angle mort signalé par le user (« on touche aux limites de ce que je comprenais à l'époque ») et que les insights récents du geometric DL permettent désormais de **nommer**.

In [6]:
// Config 3 : crossover géométrique de Moraglio sur permutations (OrderedEmbedding swap-walking).
// GeometricCrossover<int>(ordered:true, 2) auto-câble un OrderedEmbedding{IsOrdered=true} :
// la progéniture marche par swaps vers le centroïde métrique -> permutation valide à chaque pas.
// Mêmes parents initiaux reproductibles (ResetRng(7) dans Run) que baseline + îles -> comparaison équitable.
var geometric = Run("Geometric (Moraglio)", 100, 40,
    new GeometricCrossover<int>(ordered: true),
    new ReverseSequenceMutation(),
    new DefaultMetaHeuristic());
Console.WriteLine($"{geometric.Label,-26} tour final = {geometric.FinalTourLength:F1}");

Geometric (Moraglio)       tour final = 674,1


## Comparaison

On rapporte la longueur finale du meilleur tour (plus court = meilleur) et la dynamique de convergence échantillonnée. Attendu : les îles convergent plus lentement par génération (sous-populations plus petites) mais maintiennent la diversité plus longtemps, ce qui peut éviter des optima locaux. Le résultat exact dépend du tirage des villes et de la taille du problème — on le rapporte honnêtement, sans trier les victoires.

In [7]:
var results = new[] { baseline, island, geometric };
Console.WriteLine($"{"Config",-26} | {"Tour final",10} | {"vs baseline",12}");
Console.WriteLine(new string('-', 54));
double refLen = baseline.FinalTourLength;
foreach (var r in results)
{
    double impr = refLen > 0 ? (refLen - r.FinalTourLength) / refLen * 100.0 : 0.0;
    Console.WriteLine($"{r.Label,-26} | {r.FinalTourLength,10:F1} | {impr,+10:F1} %");
}

Console.WriteLine();
Console.WriteLine("Convergence (longueur du meilleur tour, échantillonnée) :");
foreach (var r in results)
{
    var samples = Enumerable.Range(0, r.BestPerGen.Length)
        .Where(i => i % 25 == 0 || i == r.BestPerGen.Length - 1)
        .Select(i => "g" + (i + 1) + "=" + r.BestPerGen[i].ToString("F0"));
    Console.WriteLine("  " + r.Label.PadRight(24) + string.Join("  ", samples));
}

Config                     | Tour final |  vs baseline


------------------------------------------------------


Baseline (panmictic)       |      440,8 |        0,0 %


Islands (4 x 10, Static)   |      513,9 |      -16,6 %


Geometric (Moraglio)       |      674,1 |      -52,9 %


Convergence (longueur du meilleur tour, échantillonnée) :


  Baseline (panmictic)    g1=909  g26=688  g51=551  g76=462  g100=441


  Islands (4 x 10, Static)g1=897  g26=709  g51=672  g76=554  g100=514


  Geometric (Moraglio)    g1=925  g26=763  g51=716  g76=693  g100=674


## Lecture

La grammaire de composition (`IslandMetaHeuristic` ici) s'est appliquée à un chromosome de permutation **sans aucune adaptation**. C'est la preuve que la couche de composition est agnostique à la représentation : elle opère sur `IChromosome`, ne lit jamais `Gene.Value`, et n'a donc pas besoin de savoir si les gènes sont des `double` (MGS-5) ou des index de villes (`int`, ici).

Les métaheuristiques géométriques (WOA, EO), elles, nécessitent un `GeometricConverter<double>` précisément parce qu'elles *font* de l'arithmétique sur les gènes — c'est leur travail (centroïde, spirale, opérateur géométrique), pas celui de la couche de composition. La séparation est nette : **la composition est agnostique, les composés géométriques sont typés**. Composer ne présuppose pas le domaine.

Le fait que le modèle insulaire batte, égalise ou perde sur ce TSP de taille 20 est secondaire ; le point est ailleurs : **la même brique structure des recherches sur des représentations disjointes**. C'est ce qui distingue un composant réutilisable d'une métaphore monolithique attachée à un type de problème.

**Nuance apportée par le crossover géométrique (Config 3).** Le constat « les composés géométriques sont typés » vaut pour leurs *opérateurs continus* (centroïde, spirale sur `double`). Mais la Config 3 montre que le **crossover géométrique lui-même** s'étend aux permutations dès qu'on lui fournit un embedding métrique (`OrderedEmbedding`, distance de swap). Ce n'est donc pas la *composition géométrique* qui est cantonnée aux `double`, c'est l'**espace métrique choisi** : un opérateur géométrique n'est pertinent que si son embedding reflète la structure du problème. Le centroïde naïf sur étiquettes d'indice en est la limite — d'où l'intérêt des embeddings permutation-équivariants (deep learning géométrique).

## Exercice 1 — Changer d'opérateur de croisement

`OrderedCrossover` (OX1) est l'opérateur canonique du TSP. GeneticSharp en fournit d'autres pour permutations : `PartiallyMappedCrossover` (PMX), `CycleCrossover` (CX), `OrderBasedCrossover` (OX2). Relancez le baseline avec `PartiallyMappedCrossover` et comparez la longueur finale du tour. La performance relative dépend-elle de la taille du problème ?

In [8]:
// Exercice 1 : comparez OrderedCrossover (OX1) vs PartiallyMappedCrossover (PMX).
// # Indice : remplacez new OrderedCrossover() par new PartiallyMappedCrossover() dans l'appel Run(...).
// # Étape 1 : appelez Run("Baseline (PMX)", 100, 40, new PartiallyMappedCrossover(), new ReverseSequenceMutation(), new DefaultMetaHeuristic()).
// # Étape 2 : affichez la longueur finale et comparez à baseline-OX1.
// TODO étudiant
Console.WriteLine($"[Exercice 1] À compléter. Référence : baseline-OX1 = {baseline.FinalTourLength:F1}.");

[Exercice 1] À compléter. Référence : baseline-OX1 = 440,8.


## Exercice 2 — Mode de migration

Le modèle insulaire supporte plusieurs `MigrationMode` : `Static`, `RandomRing`, `RandomPermutation`, `Reinforced`. Chaque mode détermine quelles îles échangent des chromosomes. Relancez la configuration îles avec `MigrationMode.RandomRing` et comparez la longueur finale à `Static`.

In [9]:
// Exercice 2 : comparez MigrationMode.Static vs MigrationMode.RandomRing.
// # Indice : recréez IslandMetaHeuristic(10, 4, new DefaultMetaHeuristic()) avec MigrationMode = MigrationMode.RandomRing.
// TODO étudiant
Console.WriteLine($"[Exercice 2] À compléter. Référence : islands-Static = {island.FinalTourLength:F1}.");

[Exercice 2] À compléter. Référence : islands-Static = 513,9.


## Exercice 3 — Topologie des îles

La partition `(islandSize, islandNb)` doit satisfaire `islandSize × islandNb = popSize`. Pour une population de 40, les partitions valides incluent `(20, 2)`, `(10, 4)`, `(5, 8)`, `(4, 10)`. Testez l'effet de la granularité — peu d'îles grandes vs beaucoup de petites — sur la diversité maintenue et la qualité de la convergence. Quelle granularité évite le mieux la convergence prématurée ?

In [10]:
// Exercice 3 : testez plusieurs partitions (islandSize, islandNb) de produit 40.
// # Indice : (20,2), (10,4), (5,8), (4,10). Relancez Run(...) avec un IslandMetaHeuristic par partition.
// # Étape 1 : bouclez sur les partitions et appelez Run pour chacune.
// # Étape 2 : affichez la longueur finale de chaque partition.
// TODO étudiant
Console.WriteLine("[Exercice 3] À compléter. Comparez les longueurs finales selon la granularité.");

[Exercice 3] À compléter. Comparez les longueurs finales selon la granularité.


## Conclusion

Le même moteur `MetaGeneticAlgorithm` et les mêmes primitives de composition (`IslandMetaHeuristic`) ont structuré une recherche sur des **permutations**, alors que MGS-5 les utilisait sur des surfaces **continues**. La couche de composition ne fait aucune hypothèse sur le type des gènes : elle orchestre des `IChromosome`. C'est l'agnosticité de représentation héritée de GeneticSharp (bit, permutation, arbre, flottant), étendue à la couche métaheuristique.

Les métaheuristiques géométriques (WOA, EO) restent attachées aux gènes `double` — non par limitation de la composition, mais parce que leur *mécanisme* (opérateur géométrique, centroïde, spirale) est de l'arithmétique sur des positions continues. La séparation est nette : **la composition est agnostique, les composés géométriques sont typés**.

Suite logique : combiner cette agnosticité avec la structuration — un modèle insulaire où chaque île utiliserait un opérateur différent (configuration hétérogène), via la surcharge `IslandMetaHeuristic(params (int, IMetaHeuristic)[])`.


**Extension : crossover géométrique de Moraglio.** La Config 3 (`GeometricCrossover<int>` + `OrderedEmbedding`) étend la géométrie aux permutations : la progéniture marche par swaps sur le segment métrique entre parents. Le résultat empirique (tour final reporté dans la comparaison ci-dessus) se lit à l'aune de la théorie — l'opérateur centroïde agit sur des étiquettes d'indice sans signification géographique, ce qui borne ce qu'un crossover géométrique *naïf* peut atteindre et motive les embeddings permutation-équivariants du deep learning géométrique.

**Références.**
- A. Moraglio & R. Poli, *Topological Interpretation of Crossover*, GECCO 2004 ; A. Moraglio, *Towards a Geometric Unification of Evolutionary Algorithms*, thèse 2007 — le cadre du crossover géométrique (segment métrique entre parents, valide en toute métrique, dont la distance de swap).
- M. Bronstein, J. Bruna, T. Cohen, P. Veličković, *Geometric Deep Learning* — https://geometricdeeplearning.com — équivariance par permutation et embeddings structurés.

[← MGS-6 Benchmarks](MGS-6-Benchmarks.ipynb) | [↑ Série MGS](README.md) | [Fork jsboige/MetaGeneticSharp →](https://github.com/jsboige/MetaGeneticSharp)